**ALGORITHMIC EVALUATION METRICS**

CODE THE EXACT MATHEMATICAL FORMULAS FOR BLEU (n-gram precision) and ROUGE-L (longest common subsequence) metrics without using external libraries.

In [19]:
import math
from collections import Counter

In [21]:
# Tokenization

# Example sentences where higher-order n-grams would normally drop to zero
reference = "the cat is sitting on the mat".split()
candidate = "the cat sat on the mat".split()

In [22]:
# Generate N-grams

def generate_ngrams(tokens, n):
    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens)-n+1)
    ]

In [23]:
# BLEU Score (With Smoothing)

def bleu_score(reference, candidate, max_n=4):
    precisions = []

    for n in range(1, max_n+1):
        ref_ngrams = Counter(generate_ngrams(reference, n))
        cand_ngrams = Counter(generate_ngrams(candidate, n))

        matches = 0
        total = sum(cand_ngrams.values())

        # If the candidate length is shorter than n, total will be 0
        if total == 0:
            precisions.append(0)
            continue

        for gram in cand_ngrams:
            matches += min(cand_ngrams[gram], ref_ngrams.get(gram, 0))

        # --- SMOOTHING IMPLEMENTATION ---
        # If there are no n-gram matches, apply a standard epsilon smoothing fraction
        # (e.g., 0.5) to prevent the entire geometric mean from collapsing to absolute zero.
        if matches == 0:
            p_n = 0.5 / total
        else:
            p_n = matches / total

        precisions.append(p_n)

    # Brevity Penalty
    c = len(candidate)
    r = len(reference)

    if c > r:
        bp = 1
    else:
        # Avoid division by zero if candidate is empty
        bp = math.exp(1 - r / c) if c > 0 else 0

    # Geometric Mean Calculation
    score = bp
    for p in precisions:
        if p == 0:
            return 0
        score *= p ** (1 / max_n)

    return score

In [24]:
# Longest Common Subsequence (LCS)

def lcs_length(a, b):
    m = len(a)
    n = len(b)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    return dp[m][n]

In [25]:
# ROUGE-L

def rouge_l(reference, candidate):
    lcs = lcs_length(reference, candidate)

    # Handle edge case where candidate or reference is empty
    if len(candidate) == 0 or len(reference) == 0:
        return 0.0, 0.0, 0.0

    precision = lcs / len(candidate)
    recall = lcs / len(reference)

    # Avoid ZeroDivisionError if there are zero overlapping words at all
    if (precision + recall) == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return precision, recall, f1

In [26]:
# Evaluation & Execution

bleu = bleu_score(reference, candidate)
p, r, f = rouge_l(reference, candidate)

print("Evaluation Results :")
print("BLEU Score        :", round(bleu, 4))
print("ROUGE-L Precision :", round(p, 4))
print("ROUGE-L Recall    :", round(r, 4))
print("ROUGE-L F1 Score  :", round(f, 4))

Evaluation Results :
BLEU Score        : 0.3216
ROUGE-L Precision : 0.8333
ROUGE-L Recall    : 0.7143
ROUGE-L F1 Score  : 0.7692
